# Native BPGD + Sinter smoke tests

This notebook upgrades the earlier guide notebook into an **actually runnable smoke-test / mini-benchmark** in the spirit of `BivariateBicycleCodeAnalysis.ipynb`.

It focuses on the new decoder family exposed through the relay repo's Sinter bridge:

- `regular_BP`
- `damped_BP`
- `mem_BP`
- `relay_BP`
- `BPGD`
- `BPGD-relay_BP`

and does two separate runs on the **same sampled detector data**:

1. **Fast / lighter-statistics mode**: aggregate performance only.
2. **Rich-statistics mode**: per-shot / per-stage details enabled, written through the custom decoder detail sidecars.

Using the same sampled batch for both modes makes the runtime comparison cleaner: differences are mostly decoder-side overhead, not Monte Carlo randomness.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sinter
import stim

from relay_bp.stim import sinter_decoders_from_specs
from relay_bp.stim.sinter.utils import write_stats

## Notebook parameters

These defaults are intentionally **small** so the notebook can be used as a decoder smoke test.

- `shots_per_circuit` is kept small.
- `max_circuits = None` means use all matched Gross-code memory-X error-rate files.
- Set `max_circuits = 3` if you want a quicker first run.


In [ ]:
repo_root = Path('.').resolve()

circuits_dir = repo_root / 'tests' / 'testdata' / 'bicycle_bivariate'
decoder_config_path = repo_root / 'configs' / 'decoder_specs_gross_144_12_12_memory_X_sinter.json'

# Match the Gross-code 144_12_12 memory-X circuits.
filename_contains = 'circuit=bicycle_bivariate_144_12_12_memory_X'

# Small smoke-test settings.
shots_per_circuit = 200
max_circuits = None   # Set to 3 for a very quick check.
seed = 12345

# Keep Sinter tiny in the optional section near the end.
sinter_smoke_shots = 200
sinter_smoke_workers = 1

notebook_data_dir = repo_root / 'notebook_data' / 'native_bpgd_sinter_smoke'
notebook_data_dir.mkdir(parents=True, exist_ok=True)

fast_results_dir = notebook_data_dir / 'fast_mode'
rich_results_dir = notebook_data_dir / 'rich_mode'
collect_results_dir = notebook_data_dir / 'sinter_collect_smoke'
for p in [fast_results_dir, rich_results_dir, collect_results_dir]:
    p.mkdir(parents=True, exist_ok=True)

print('repo_root =', repo_root)
print('circuits_dir exists =', circuits_dir.exists())
print('decoder_config exists =', decoder_config_path.exists())

## Helper utilities

These follow the same repo conventions used by the new Gross-code Sinter runner, but keep everything notebook-friendly.


In [ ]:
def parse_name_metadata(path: Path) -> dict:
    metadata = {'stim_path': str(path)}
    for part in path.stem.split(','):
        if '=' not in part:
            continue
        key, value = part.split('=', 1)
        metadata[key] = value
    if 'error_rate' in metadata:
        try:
            metadata['p'] = float(metadata['error_rate'])
        except ValueError:
            pass
    if 'distance' in metadata:
        try:
            metadata['d'] = int(metadata['distance'])
        except ValueError:
            pass
    if 'rounds' in metadata:
        try:
            metadata['r'] = int(metadata['rounds'])
        except ValueError:
            pass
    return metadata


def find_circuits(circuits_dir: Path, filename_contains: str, max_circuits: int | None = None) -> list[Path]:
    paths = sorted(circuits_dir.glob('*.stim'))
    needles = [needle.strip() for needle in filename_contains.split(',') if needle.strip()]
    if needles:
        paths = [path for path in paths if all(needle in path.name for needle in needles)]
    if max_circuits is not None:
        paths = paths[:max_circuits]
    if not paths:
        raise FileNotFoundError(
            f'No .stim files matched in {circuits_dir!s} with filter {filename_contains!r}'
        )
    return paths


def load_decoder_specs(path: Path) -> list[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        specs = json.load(f)
    if not isinstance(specs, list):
        raise TypeError('Decoder config must contain a JSON list of decoder specs.')
    return specs


def unpack_bit_packed(bits_b8: np.ndarray, count: int) -> np.ndarray:
    return np.unpackbits(bits_b8, axis=1, bitorder='little', count=count).astype(np.uint8)


def load_detail_jsonl(details_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(details_dir.glob('*.jsonl')):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)

## Load the decoder specs and matched circuits

In [ ]:
specs = load_decoder_specs(decoder_config_path)
matched_circuits = find_circuits(circuits_dir, filename_contains, max_circuits=max_circuits)

decoder_names = [spec['name'] for spec in specs]
metadata_rows = [parse_name_metadata(path) for path in matched_circuits]

print('Decoders:')
for name in decoder_names:
    print('  -', name)

print('
Matched circuits:', len(matched_circuits))
pd.DataFrame(metadata_rows)[['circuit', 'p', 'basis', 'noise_model', 'stim_path']]

## Build two decoder registries

- **Fast mode** keeps the original Sinter-style observable prediction path and does not ask for detailed per-shot decode records.
- **Rich mode** enables `include_decode_result=True` and writes JSONL detail sidecars through the compiled decoder wrapper.


In [ ]:
# Reset detail directories so reruns stay easy to interpret.
for detail_dir in [fast_results_dir / 'details', rich_results_dir / 'details']:
    if detail_dir.exists():
        shutil.rmtree(detail_dir)
    detail_dir.mkdir(parents=True, exist_ok=True)

fast_custom_decoders = sinter_decoders_from_specs(
    specs,
    include_decode_result=False,
    details_dir=None,
)

rich_custom_decoders = sinter_decoders_from_specs(
    specs,
    include_decode_result=True,
    details_dir=str(rich_results_dir / 'details'),
)

sorted(fast_custom_decoders.keys())

## Pre-sample a fixed detector batch for each circuit

This is the key notebook design choice for the runtime comparison.

Instead of letting each mode re-sample independently, we sample **once** with a fixed `seed` and reuse the exact same detector and observable data for both runs. That way the fast-vs-rich timing comparison is not blurred by Monte Carlo randomness.

Stim supports detector sampling via `compile_detector_sampler()` and `sample(..., separate_observables=True)`. citeturn744688search0turn744688search1


In [ ]:
sample_cache = {}

for circuit_path in matched_circuits:
    circuit = stim.Circuit.from_file(circuit_path)
    sampler = circuit.compile_detector_sampler(seed=seed)
    dets_b8, obs_b8 = sampler.sample(
        shots=shots_per_circuit,
        separate_observables=True,
        bit_packed=True,
    )
    sample_cache[circuit_path] = {
        'circuit': circuit,
        'detector_error_model': circuit.detector_error_model(),
        'metadata': parse_name_metadata(circuit_path),
        'dets_b8': dets_b8,
        'obs_b8': obs_b8,
        'num_observables': circuit.num_observables,
    }

print(f'Prepared fixed sampled batches for {len(sample_cache)} circuits with seed={seed}.')

## Benchmark helper

This helper mirrors the Sinter compiled-decoder interface directly:

1. compile the custom decoder for the circuit's DEM,
2. decode the pre-sampled detector shots,
3. compare predicted observables to the sampled actual observables,
4. record runtime and logical failure counts.

This is **not** a replacement for `sinter.collect(...)`; it is a notebook-friendly way to compare fast-vs-rich decoder overhead on exactly the same data.


In [ ]:
def benchmark_decoder_registry(
    sample_cache: dict,
    custom_decoders: dict,
    mode_name: str,
) -> pd.DataFrame:
    rows = []
    for circuit_path, cached in sample_cache.items():
        dem = cached['detector_error_model']
        dets_b8 = cached['dets_b8']
        actual_obs_b8 = cached['obs_b8']
        num_obs = cached['num_observables']
        actual_obs = unpack_bit_packed(actual_obs_b8, num_obs)
        metadata = cached['metadata']

        for decoder_name, decoder in custom_decoders.items():
            compiled = decoder.compile_decoder_for_dem(dem=dem)
            t0 = time.perf_counter()
            predicted_b8 = compiled.decode_shots_bit_packed(
                bit_packed_detection_event_data=dets_b8,
            )
            elapsed = time.perf_counter() - t0
            predicted_obs = unpack_bit_packed(predicted_b8, num_obs)
            logical_failures = int(np.any(predicted_obs != actual_obs, axis=1).sum())
            rows.append(
                {
                    'mode': mode_name,
                    'decoder': decoder_name,
                    'stim_path': str(circuit_path),
                    'circuit': metadata.get('circuit', ''),
                    'p': metadata.get('p', np.nan),
                    'basis': metadata.get('basis', ''),
                    'shots': int(actual_obs.shape[0]),
                    'logical_failures': logical_failures,
                    'logical_error_rate': logical_failures / max(1, int(actual_obs.shape[0])),
                    'runtime_sec': elapsed,
                    'shots_per_sec': int(actual_obs.shape[0]) / elapsed if elapsed > 0 else np.nan,
                }
            )
    return pd.DataFrame(rows)


def summarise_runtime(df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df.groupby(['mode', 'decoder'], as_index=False)
        .agg(
            circuits=('stim_path', 'nunique'),
            shots=('shots', 'sum'),
            logical_failures=('logical_failures', 'sum'),
            runtime_sec=('runtime_sec', 'sum'),
        )
    )
    grouped['logical_error_rate'] = grouped['logical_failures'] / grouped['shots']
    grouped['shots_per_sec'] = grouped['shots'] / grouped['runtime_sec']
    return grouped.sort_values(['decoder', 'mode']).reset_index(drop=True)

## Run the fast and rich decoder passes

Both passes use the **same sampled detector data** and the **same `seed`**, so differences here are mostly decoder-side overhead and detail-recording overhead.


In [ ]:
fast_df = benchmark_decoder_registry(sample_cache, fast_custom_decoders, mode_name='fast')
rich_df = benchmark_decoder_registry(sample_cache, rich_custom_decoders, mode_name='rich')

benchmark_df = pd.concat([fast_df, rich_df], ignore_index=True)
runtime_summary_df = summarise_runtime(benchmark_df)

fast_csv = fast_results_dir / 'fast_mode_decoder_benchmark.csv'
rich_csv = rich_results_dir / 'rich_mode_decoder_benchmark.csv'
runtime_csv = notebook_data_dir / 'fast_vs_rich_runtime_summary.csv'

fast_df.to_csv(fast_csv, index=False)
rich_df.to_csv(rich_csv, index=False)
runtime_summary_df.to_csv(runtime_csv, index=False)

runtime_summary_df

## Compare runtime overhead and small-shot decoder performance

In [ ]:
runtime_pivot = runtime_summary_df.pivot(index='decoder', columns='mode', values='runtime_sec')
throughput_pivot = runtime_summary_df.pivot(index='decoder', columns='mode', values='shots_per_sec')
ler_pivot = runtime_summary_df.pivot(index='decoder', columns='mode', values='logical_error_rate')

comparison_df = runtime_summary_df.pivot(index='decoder', columns='mode', values=['runtime_sec', 'shots_per_sec', 'logical_error_rate'])
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

runtime_plot_df = runtime_summary_df.pivot(index='decoder', columns='mode', values='runtime_sec')
runtime_plot_df.plot(kind='bar', ax=axes[0])
axes[0].set_ylabel('Total runtime [s]')
axes[0].set_title('Fast vs rich runtime')
axes[0].grid(True, axis='y')

ler_plot_df = runtime_summary_df.pivot(index='decoder', columns='mode', values='logical_error_rate')
ler_plot_df.plot(kind='bar', ax=axes[1])
axes[1].set_ylabel('Logical error rate')
axes[1].set_title('Small-shot logical error rate')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

## Inspect the rich per-shot / per-stage detail sidecars

The richer mode writes JSONL detail files through the Sinter compiled decoder wrapper. Those records include the statistics discussed throughout the chat:

- total iterations,
- convergence,
- fallback usage,
- stage names,
- stage-wise iteration counts.


In [ ]:
detail_df = load_detail_jsonl(rich_results_dir / 'details')
print('detail rows =', len(detail_df))
detail_df.head()

In [ ]:
if not detail_df.empty:
    detail_summary_df = (
        detail_df.groupby('decoder', as_index=False)
        .agg(
            shots=('decoder', 'size'),
            mean_iterations=('iterations', 'mean'),
            convergence_rate=('converged', 'mean'),
            fallback_rate=('fallback_used', lambda s: pd.Series(s).fillna(False).astype(bool).mean()),
        )
        .sort_values('decoder')
        .reset_index(drop=True)
    )
else:
    detail_summary_df = pd.DataFrame()

detail_summary_df

In [ ]:
if not detail_df.empty:
    stage_expanded = detail_df[['decoder', 'stage_names', 'stage_iterations', 'stage_converged']].copy()
    stage_expanded = stage_expanded.explode(['stage_names', 'stage_iterations', 'stage_converged'], ignore_index=True)
    stage_expanded = stage_expanded.rename(
        columns={
            'stage_names': 'stage_name',
            'stage_iterations': 'stage_iterations',
            'stage_converged': 'stage_converged',
        }
    )
    stage_summary_df = (
        stage_expanded.groupby(['decoder', 'stage_name'], as_index=False)
        .agg(
            mean_stage_iterations=('stage_iterations', 'mean'),
            stage_convergence_rate=('stage_converged', 'mean'),
            observations=('stage_name', 'size'),
        )
        .sort_values(['decoder', 'stage_name'])
        .reset_index(drop=True)
    )
else:
    stage_summary_df = pd.DataFrame()

stage_summary_df

## Optional: tiny `sinter.collect(...)` smoke run

The benchmark above is the cleanest way to compare fast-vs-rich overhead on the **same sampled data**.

This cell adds a tiny repository-style `sinter.collect(...)` run, closer to the original relay notebooks. It uses the richer decoder registry, a tiny number of shots, and a resumable CSV path.

Sinter's collection API supports `custom_decoders` and `save_resume_filepath`, which is why it is the right orchestration layer for the repo-style experiment scripts. citeturn744688search2


In [ ]:
tiny_tasks = []
for circuit_path in matched_circuits[: min(len(matched_circuits), 2)]:
    circuit = stim.Circuit.from_file(circuit_path)
    tiny_tasks.append(
        sinter.Task(
            circuit=circuit,
            detector_error_model=circuit.detector_error_model(),
            json_metadata=parse_name_metadata(circuit_path),
            collection_options=sinter.CollectionOptions(max_shots=sinter_smoke_shots),
        )
    )

resume_csv = collect_results_dir / 'native_bpgd_sinter_smoke_resume.csv'
summary_csv = collect_results_dir / 'native_bpgd_sinter_smoke_summary.csv'

smoke_samples = sinter.collect(
    tasks=tiny_tasks,
    decoders=list(rich_custom_decoders.keys()),
    custom_decoders=rich_custom_decoders,
    num_workers=sinter_smoke_workers,
    save_resume_filepath=resume_csv,
    print_progress=False,
)

write_stats(smoke_samples, summary_csv)

print('resume_csv =', resume_csv)
print('summary_csv =', summary_csv)
pd.DataFrame([
    {
        'decoder': s.decoder,
        'shots': s.shots,
        'errors': s.errors,
        'logical_error_rate': s.errors / s.shots if s.shots else np.nan,
        'seconds': getattr(s, 'seconds', np.nan),
        **dict(getattr(s, 'json_metadata', {}) or {}),
    }
    for s in smoke_samples
])

## Files written by this notebook

After a successful run, look here:

- `notebook_data/native_bpgd_sinter_smoke/fast_mode/fast_mode_decoder_benchmark.csv`
- `notebook_data/native_bpgd_sinter_smoke/rich_mode/rich_mode_decoder_benchmark.csv`
- `notebook_data/native_bpgd_sinter_smoke/fast_vs_rich_runtime_summary.csv`
- `notebook_data/native_bpgd_sinter_smoke/rich_mode/details/*.jsonl`
- `notebook_data/native_bpgd_sinter_smoke/sinter_collect_smoke/*.csv`

This gives you:

- quick small-shot decoder performance,
- a controlled fast-vs-rich runtime comparison,
- and a tiny repo-style Sinter smoke run.
